In [1]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00


In [2]:
#easy question:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Create a 2-qubit circuit
qc = QuantumCircuit(2)

# Put the control qubit in |1>
qc.x(0)

# Apply CNOT
qc.cx(0, 1)

# Get final statevector
state = Statevector.from_instruction(qc)

print("CNOT Gate Circuit:")
print(qc)

print("\nFinal Statevector:")
print(state)

print("\nProbabilities:")
print(state.probabilities())

CNOT Gate Circuit:
     ┌───┐     
q_0: ┤ X ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Final Statevector:
Statevector([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
            dims=(2, 2))

Probabilities:
[0. 0. 0. 1.]


In [5]:
#medium
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Create a 2-qubit circuit with 2 classical bits
qc = QuantumCircuit(2, 2)

# Create Bell state
qc.h(0)
qc.cx(0, 1)

# Measure both qubits
qc.measure([0, 1], [0, 1])

# Display the circuit
print("Bell State Circuit:")
print(qc)

# Run the circuit for 1024 shots
simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()

# Get measurement results
counts = result.get_counts()

print("\nMeasurement Results:")
print(counts)

# Verify that only 00 and 11 occurred
print("\nVerification:")

if set(counts.keys()).issubset({"00", "11"}):
    print("Only 00 and 11 outcomes were observed.")
else:
    print("Unexpected outcomes were observed.")

Bell State Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Measurement Results:
{'00': 498, '11': 526}

Verification:
Only 00 and 11 outcomes were observed.


In [7]:
#hard question
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Create a 3-qubit circuit with 3 classical bits
qc = QuantumCircuit(3, 3)

# Create GHZ state
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)

# Measure all three qubits
qc.measure([0, 1, 2], [0, 1, 2])

# Display the circuit
print("3-Qubit GHZ Circuit:")
print(qc)

# Run the circuit for 1024 shots
simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()

# Get measurement results
counts = result.get_counts()

print("\nMeasurement Results:")
print(counts)

# Verify perfect three-way correlation
print("\nVerification:")

if set(counts.keys()).issubset({"000", "111"}):
    print("Perfect three-way correlation confirmed.")
    print("Only 000 and 111 outcomes were observed.")
else:
    print("Unexpected outcomes were observed.")

3-Qubit GHZ Circuit:
     ┌───┐             ┌─┐   
q_0: ┤ H ├──■────■─────┤M├───
     └───┘┌─┴─┐  │  ┌─┐└╥┘   
q_1: ─────┤ X ├──┼──┤M├─╫────
          └───┘┌─┴─┐└╥┘ ║ ┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     1  0  2 

Measurement Results:
{'000': 516, '111': 508}

Verification:
Perfect three-way correlation confirmed.
Only 000 and 111 outcomes were observed.


In [8]:
#real world question

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# ---------------------------------------------------------
# Full Adder using CNOT and Toffoli gates
# Inputs  : A, B, Cin
# Outputs : Sum, Carry
# ---------------------------------------------------------

def full_adder(A, B, Cin):
    # 5 qubits:
    # q0 = A
    # q1 = B
    # q2 = Cin
    # q3 = Sum
    # q4 = Carry
    qc = QuantumCircuit(5, 2)

    # Set input values
    if A == 1:
        qc.x(0)
    if B == 1:
        qc.x(1)
    if Cin == 1:
        qc.x(2)

    # ---------------- SUM ----------------
    # Sum = A XOR B XOR Cin
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # ---------------- CARRY ----------------
    # Carry = AB XOR Cin(A XOR B)

    # First calculate AB
    qc.ccx(0, 1, 4)

    # Transform B temporarily to A XOR B
    qc.cx(0, 1)

    # Calculate Cin(A XOR B)
    qc.ccx(1, 2, 4)

    # Restore B
    qc.cx(0, 1)

    # Measure Sum and Carry
    qc.measure(3, 0)
    qc.measure(4, 1)

    return qc


# ---------------------------------------------------------
# Test the Full Adder for all possible inputs
# ---------------------------------------------------------

simulator = AerSimulator()

print("FULL ADDER USING CNOT AND TOFFOLI GATES")
print("=" * 50)

for A in [0, 1]:
    for B in [0, 1]:
        for Cin in [0, 1]:

            qc = full_adder(A, B, Cin)

            result = simulator.run(qc, shots=1).result()
            counts = result.get_counts()

            # Get output
            output = list(counts.keys())[0]

            print(
                f"A = {A}, B = {B}, Cin = {Cin} "
                f"--> Sum = {output[0]}, Carry = {output[1]}"
            )


# ---------------------------------------------------------
# Display one example circuit
# ---------------------------------------------------------

print("\nExample Circuit for A=1, B=1, Cin=1:")
example = full_adder(1, 1, 1)
print(example)

FULL ADDER USING CNOT AND TOFFOLI GATES
A = 0, B = 0, Cin = 0 --> Sum = 0, Carry = 0
A = 0, B = 0, Cin = 1 --> Sum = 0, Carry = 1
A = 0, B = 1, Cin = 0 --> Sum = 0, Carry = 1
A = 0, B = 1, Cin = 1 --> Sum = 1, Carry = 0
A = 1, B = 0, Cin = 0 --> Sum = 0, Carry = 1
A = 1, B = 0, Cin = 1 --> Sum = 1, Carry = 0
A = 1, B = 1, Cin = 0 --> Sum = 1, Carry = 0
A = 1, B = 1, Cin = 1 --> Sum = 1, Carry = 1

Example Circuit for A=1, B=1, Cin=1:
     ┌───┐                                   
q_0: ┤ X ├──■──────────────■────■─────────■──
     ├───┤  │              │  ┌─┴─┐     ┌─┴─┐
q_1: ┤ X ├──┼────■─────────■──┤ X ├──■──┤ X ├
     ├───┤  │    │         │  └───┘  │  └───┘
q_2: ┤ X ├──┼────┼────■────┼─────────■───────
     └───┘┌─┴─┐┌─┴─┐┌─┴─┐  │   ┌─┐   │       
q_3: ─────┤ X ├┤ X ├┤ X ├──┼───┤M├───┼───────
          └───┘└───┘└───┘┌─┴─┐ └╥┘ ┌─┴─┐ ┌─┐ 
q_4: ────────────────────┤ X ├──╫──┤ X ├─┤M├─
                         └───┘  ║  └───┘ └╥┘ 
c: 2/═══════════════════════════╩═════════╩══
          

In [9]:
#challenge question

In [10]:
# Install Qiskit (run once)

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# Create a 2-qubit circuit
qc = QuantumCircuit(2)

# ---------------------------------------------------------
# Create an arbitrary entangled state
# |ψ> = √3/2 |00> + 1/2 |11>
# ---------------------------------------------------------

# Apply Ry rotation to the first qubit
theta = np.pi / 3
qc.ry(theta, 0)

# Entangle the two qubits using CNOT
qc.cx(0, 1)

# ---------------------------------------------------------
# Display the circuit
# ---------------------------------------------------------

print("Quantum Circuit:")
print(qc)

# ---------------------------------------------------------
# State tomography / Statevector inspection
# ---------------------------------------------------------

state = Statevector.from_instruction(qc)

print("\nFinal Statevector:")
print(state)

# Display probabilities
probabilities = state.probabilities()

print("\nProbabilities of computational basis states:")
print("|00> :", round(probabilities[0], 4))
print("|01> :", round(probabilities[1], 4))
print("|10> :", round(probabilities[2], 4))
print("|11> :", round(probabilities[3], 4))

# ---------------------------------------------------------
# Verify the state
# ---------------------------------------------------------

expected_state = Statevector([
    np.sqrt(3) / 2,
    0,
    0,
    1 / 2
])

print("\nExpected Statevector:")
print(expected_state)

print("\nVerification:")

if state.equiv(expected_state):
    print("The generated state matches the chosen arbitrary entangled state.")
else:
    print("The generated state does not match the expected state.")

Quantum Circuit:
     ┌─────────┐     
q_0: ┤ Ry(π/3) ├──■──
     └─────────┘┌─┴─┐
q_1: ───────────┤ X ├
                └───┘

Final Statevector:
Statevector([0.8660254+0.j, 0.       +0.j, 0.       +0.j, 0.5      +0.j],
            dims=(2, 2))

Probabilities of computational basis states:
|00> : 0.75
|01> : 0.0
|10> : 0.0
|11> : 0.25

Expected Statevector:
Statevector([0.8660254+0.j, 0.       +0.j, 0.       +0.j, 0.5      +0.j],
            dims=(2, 2))

Verification:
The generated state matches the chosen arbitrary entangled state.
